# 5G-NIDD 80/20 stratified split + balancing on train only

This notebook will:
1. Load `Encoded.csv`
2. Show the original label distribution
3. Create a **stratified 80/20 split per class** (each attack type + benign split individually)
4. Perform **upsampling/downsampling only on the 80% train pool** to produce a **50/50 balanced training set**
5. Save outputs into `Datasets/`
6. Print a final summary of dataset sizes and per-class counts.


In [ ]:
import pandas as pd
import numpy as np
import os
import re

# CONFIG
CSV_PATH     = "Encoded.csv"
ATTACK_COL   = "Attack Type"
BENIGN_NAME  = "Benign"
N_PER_MAL    = 10000
SEED         = 42
TEST_FRAC    = 0.20  # 20% test, 80% train pool

OUT_DIR = "Datasets"
os.makedirs(OUT_DIR, exist_ok=True)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)


In [ ]:
# Load
df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())


In [ ]:
# Original distribution
counts = df[ATTACK_COL].value_counts(dropna=False)
display(counts)

percent = (counts / len(df) * 100).round(2)
display(pd.DataFrame({"count": counts, "percent": percent}))


## Helpers


In [ ]:
def safe_filename(name: str) -> str:
    name = str(name).strip()
    name = re.sub(r"[^\w\-\.]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unknown"

def sample_to_n(df_in: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """Sample df_in to exactly n rows, with replacement if needed."""
    replace = len(df_in) < n
    return df_in.sample(n=n, replace=replace, random_state=seed)

def stratified_train_test_split_by_label(df: pd.DataFrame, label_col: str, test_frac: float, seed: int):
    """
    Split df into train/test by splitting each label group separately.
    For each label:
      - shuffle once
      - take first n_test rows as test
      - remaining rows as train
    This guarantees no overlap between train and test.
    """
    train_parts, test_parts = [], []

    labels = df[label_col].dropna().astype(str).unique().tolist()
    labels = sorted(labels)

    for lbl in labels:
        g = df[df[label_col].astype(str) == lbl]
        n = len(g)
        if n == 0:
            continue

        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)

        n_test = int(round(n * test_frac))
        if n >= 2:
            n_test = max(1, min(n - 1, n_test))
        else:
            n_test = 0

        g_test = g.iloc[:n_test].copy()
        g_train = g.iloc[n_test:].copy()

        test_parts.append(g_test)
        train_parts.append(g_train)

        print(f"{lbl:20s} total={n:7d} -> train={len(g_train):7d}, test={len(g_test):7d}")

    df_train = pd.concat(train_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df_test  = pd.concat(test_parts,  axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df_train, df_test


## 80/20 split per class


In [ ]:
df_train_pool, df_test = stratified_train_test_split_by_label(
    df=df,
    label_col=ATTACK_COL,
    test_frac=TEST_FRAC,
    seed=SEED
)

print("\nTrain pool shape:", df_train_pool.shape)
print("Test shape:", df_test.shape)


## Balance ONLY the train pool (50/50)


In [ ]:
# Split benign vs malicious within the TRAIN POOL
df_ben = df_train_pool[df_train_pool[ATTACK_COL].astype(str) == BENIGN_NAME].copy()
df_mal = df_train_pool[df_train_pool[ATTACK_COL].astype(str) != BENIGN_NAME].copy()

mal_types = sorted(df_mal[ATTACK_COL].dropna().astype(str).unique().tolist())
print("Malicious attack types (train pool):", mal_types)
print("Count malicious types (train pool):", len(mal_types))

# Balance malicious per type -> N_PER_MAL
parts_mal = []
mal_balanced_by_type = {}

for t in mal_types:
    sub = df_mal[df_mal[ATTACK_COL].astype(str) == t]
    n = len(sub)

    sub_bal = sample_to_n(sub, N_PER_MAL, SEED).reset_index(drop=True)
    mal_balanced_by_type[t] = sub_bal
    parts_mal.append(sub_bal)

    print(f"{t:15s} original={n:7d} -> balanced={len(sub_bal):7d} (replace={n < N_PER_MAL})")

df_mal_bal = pd.concat(parts_mal, axis=0).reset_index(drop=True)

# Benign should match TOTAL malicious count (for 50/50 overall in TRAIN)
target_benign = len(df_mal_bal)
df_ben_bal = sample_to_n(df_ben, target_benign, SEED).reset_index(drop=True)

print("\nBenign (train pool) original:", len(df_ben), "-> target:", target_benign,
      f"(replace={len(df_ben) < target_benign})")

# Final balanced TRAIN dataset (50/50)
df_train_balanced = (
    pd.concat([df_ben_bal, df_mal_bal], axis=0)
      .sample(frac=1.0, random_state=SEED)
      .reset_index(drop=True)
)

print("\nFinal balanced TRAIN dataset shape:", df_train_balanced.shape)

# Distribution sanity check
counts_train_bal = df_train_balanced[ATTACK_COL].value_counts(dropna=False)
percent_train_bal = (counts_train_bal / len(df_train_balanced) * 100).round(2)
display(pd.DataFrame({"count": counts_train_bal, "percent": percent_train_bal}))


## Save outputs


In [ ]:
# 1) Balanced train dataset
out_train_bal = os.path.join(OUT_DIR, f"Encoded_train_balanced_{len(df_train_balanced)}.csv")
df_train_balanced.to_csv(out_train_bal, index=False)
print("Saved balanced train:", out_train_bal)

# 2) Benign dataset for balancing (train)
TRAIN_PER_CLASS_DIR = os.path.join(OUT_DIR, "train_per_class")
os.makedirs(TRAIN_PER_CLASS_DIR, exist_ok=True)
out_ben = os.path.join(TRAIN_PER_CLASS_DIR, f"Encoded_train_{safe_filename(BENIGN_NAME)}_{target_benign}.csv")
df_ben_bal.to_csv(out_ben, index=False)
print("Saved benign train:", out_ben)

# 3) One train dataset per malicious type (train-balanced, each N_PER_MAL)
for t, df_t in mal_balanced_by_type.items():
    out_t = os.path.join(TRAIN_PER_CLASS_DIR, f"Encoded_train_{safe_filename(t)}_{N_PER_MAL}.csv")
    df_t.to_csv(out_t, index=False)

print(f"Saved {len(mal_balanced_by_type)} malicious per-type train CSVs into: {TRAIN_PER_CLASS_DIR}")

# 4) Test CSV per class (attack type + benign)
TEST_PER_CLASS_DIR = os.path.join(OUT_DIR, "test_per_class")
os.makedirs(TEST_PER_CLASS_DIR, exist_ok=True)

df_test[ATTACK_COL] = df_test[ATTACK_COL].astype(str)

test_labels = sorted(df_test[ATTACK_COL].dropna().astype(str).unique().tolist())
print("Test labels:", test_labels)

saved_paths = []
for lbl in test_labels:
    sub = df_test[df_test[ATTACK_COL].astype(str) == lbl].copy()
    out_lbl = os.path.join(TEST_PER_CLASS_DIR, f"Encoded_test_{safe_filename(lbl)}_{len(sub)}.csv")
    sub.to_csv(out_lbl, index=False)
    saved_paths.append((lbl, len(sub), out_lbl))

print(f"\nSaved {len(saved_paths)} per-class TEST CSVs into: {TEST_PER_CLASS_DIR}")
display(pd.DataFrame(saved_paths, columns=["attack_type", "rows", "path"]))


# 5) Final test dataset
out_test = os.path.join(OUT_DIR, f"Encoded_test_20pct_{len(df_test)}.csv")
df_test.to_csv(out_test, index=False)
print("Saved shuffled test:", out_test)


## Final summary


In [ ]:
def summarize(df_in: pd.DataFrame, name: str, label_col: str):
    print(f"\n=== {name} ===")
    print("Rows:", len(df_in), "| Columns:", df_in.shape[1])
    vc = df_in[label_col].value_counts(dropna=False)
    pct = (vc / len(df_in) * 100).round(2) if len(df_in) else vc
    display(pd.DataFrame({"count": vc, "percent": pct}))

summarize(df, "Original", ATTACK_COL)
summarize(df_train_pool, "Train pool (80%)", ATTACK_COL)
summarize(df_test, "Test (20%)", ATTACK_COL)
summarize(df_train_balanced, "Train balanced (50/50)", ATTACK_COL)

print("\nPer-type balanced files created (train):")
display(pd.DataFrame({
    "attack_type": list(mal_balanced_by_type.keys()),
    "rows": [len(x) for x in mal_balanced_by_type.values()]
}).sort_values("attack_type").reset_index(drop=True))

print("\nBenign rows used in balanced train:", len(df_ben_bal))
print("Total malicious rows used in balanced train:", len(df_mal_bal))
